In [ ]:
# pyright: reportUnusedImport=false, reportMissingImports=false, reportMissingModuleSource=false
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import seaborn as sb
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, Rescaling
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

In [ ]:
normal_cells=os.listdir('brain_tumor_dataset/no/')
tumor_cells=os.listdir('brain_tumor_dataset/yes/')

# total images in each folder

In [ ]:
print('Number of normal cells: ',len(normal_cells))
print('Number of tumor cells: ',len(tumor_cells))

In [ ]:
TRAIN_DIR='brain_tumor_dataset'

# image resize 

In [ ]:
# Instantiate the dataset
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(128, 128),
    batch_size=32,
    label_mode='binary',
    seed=100
)

# Check the type    
dataset_type = type(train_dataset)
print(f'train_dataset inherits from tf.data.Dataset: {issubclass(dataset_type, tf.data.Dataset)}')

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    image_size=(128, 128),
    batch_size=32,
    label_mode='binary',
    seed=100
)
dataset_type = type(train_dataset)
print(f'train_dataset inherits from tf.data.Dataset: {issubclass(dataset_type, tf.data.Dataset)}')

# validating the image shapes

In [ ]:
# check the image shape and labels
for images, labels in train_dataset.take(1):
    print(f'Image batch shape: {images.shape}')
    print(f'Label batch shape: {labels.shape}')

In [ ]:
# genrate augmented images
data_augumentation=ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    vertical_flip=True,
    horizontal_flip=True
)

In [ ]:
rescale=Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (rescale(x), y))
val_ds = val_ds.map(lambda x, y: (rescale(x), y))

# improving performance

In [ ]:
train_dataset=(
    train_dataset.cache()
    .shuffle(100).
    prefetch(buffer_size=tf.data.AUTOTUNE)
)

In [ ]:
val_ds=(
    val_ds.cache()
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

# defining early stop at 82 percent

In [ ]:
class EarlyStopping(tf.keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs=None):
        if logs.get("accuracy")>0.82:
            print("\nReached 82% accuracy so cancelling training!")
            self.model.stop_training=True

In [ ]:
model=Sequential([
    tf.keras.layers.Input(shape=(128,128,3)),
    Conv2D(16,3,activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(32,3,activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64,3,activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(128,3,activation='relu'),
    MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    Dense(256,activation='relu'),
    Dense(1,activation='sigmoid')


])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
    metrics=['accuracy']
)

# using pre-build early stopping
using validation set also for fitting

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',  
            patience=5,
            restore_best_weights=True
        )
    ]
    )

In [ ]:

import matplotlib.pyplot as plt
try:
    h = history.history if hasattr(history, 'history') else history
    epochs = range(1, len(next(iter(h.values()))) + 1)
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    if 'accuracy' in h or 'acc' in h:
        acc = h.get('accuracy') or h.get('acc')
        plt.plot(epochs, acc, 'go-', label='Train Acc')
    if 'val_accuracy' in h or 'val_acc' in h:
        val_acc = h.get('val_accuracy') or h.get('val_acc')
        plt.plot(epochs, val_acc, 'bo-', label='Val Acc')
    plt.title('Accuracy')
    plt.xlabel('Epochs')
    plt.legend()
    plt.subplot(1,2,2)
    if 'loss' in h:
        plt.plot(epochs, h['loss'], 'go-', label='Train Loss')
    if 'val_loss' in h:
        plt.plot(epochs, h['val_loss'], 'bo-', label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epochs')
    plt.legend()
    plt.show()
except Exception as e:
    print('Plot failed:', e)
    print('Ensure `history` is the return value from `model.fit(...)`')

In [ ]:
Test_Dir="test_images"

In [ ]:
test_dataset = tf.keras.utils.image_dataset_from_directory(
    Test_Dir,
    image_size=(128, 128),
    batch_size=32,
    label_mode='binary',
    seed=100
)

In [ ]:
model.evaluate(test_dataset)

In [ ]:
model.predict(test_dataset)

# upload images 

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from PIL import Image


uploader = widgets.FileUpload(
    accept='image/*',   # Accept all image types
    multiple=False,     # Single file upload
    description='Upload Image',
    button_style='info' # Blue color
)
output = widgets.Output()

def on_predict_clicked(b):
    with output:
        clear_output()
        
        
        if not uploader.value:
            print(" Please upload an image first.")
            return
            
        print("Processing...")
        
        
        # Support both dict (new ipywidgets) and list/tuple formats
        if isinstance(uploader.value, dict):
            uploaded_file = next(iter(uploader.value.values()))
        else:
            uploaded_file = uploader.value[0]

        # filename may be at top-level or under metadata depending on widget/version
        filename = uploaded_file.get('name') or (uploaded_file.get('metadata') or {}).get('name', 'uploaded_image')
        content = uploaded_file.get('content') or uploaded_file.get('data')
        
        
        image = Image.open(io.BytesIO(content)).convert('RGB')
        display_img = image.copy()
        display_img.thumbnail((200, 200)) 
        
        
        image = image.resize((128, 128))
        image_array = np.array(image) / 255.0
        image_array = np.expand_dims(image_array, axis=0)

        
        prediction = model.predict(image_array, verbose=0)[0][0]
        label = 'Tumor' if prediction >= 0.5 else 'Normal'
        confidence = prediction if prediction >= 0.5 else 1 - prediction

        
        display(display_img)
        print(f'File: {filename}')
        print(f'Prediction: {label}')
        print(f'Confidence: {confidence:.2%}')


predict_btn = widgets.Button(
    description='Predict Tumor',
    button_style='success', 
    tooltip='Click to analyze the uploaded image'
)
predict_btn.on_click(on_predict_clicked)


print("--- Brain Tumor Classifier ---")
display(widgets.VBox([uploader, predict_btn, output]))


### wraping our model in .keras(.h5) to consume at backend fast api

In [ ]:
model.save('brain_tumor_classifier.keras')